## 0) Setup


In [88]:
import os
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

# Paths:
DATA_DIR = os.path.join('..', 'data')
OUT_DIR  = os.path.join('..', 'output')
os.makedirs(OUT_DIR, exist_ok=True)

NACC_MAIN_PATH = os.path.join(DATA_DIR, 'investigator_nacc72.csv')

USE_ONLY_V3 = False   # True => keep only FORMVER in {3, 3.2}

# Horizon sweep range
MIN_H = 3
MAX_H = 12  


## 1) Load data + construct visit_date
building `visit_date` from `VISITYR`, `VISITMO`, `VISITDAY` with sensible defaults.


In [89]:
df = pd.read_csv(NACC_MAIN_PATH, low_memory=False)
df['NACCID'] = df['NACCID'].astype(str)

def make_visit_date(df_: pd.DataFrame) -> pd.Series:
    y = pd.to_numeric(df_['VISITYR'], errors='coerce')
    m = pd.to_numeric(df_['VISITMO'], errors='coerce').fillna(6).astype(int).clip(1, 12)
    d = pd.to_numeric(df_['VISITDAY'], errors='coerce').fillna(15).astype(int).clip(1, 28)
    return pd.to_datetime(dict(year=y, month=m, day=d), errors='coerce')

df['visit_date'] = make_visit_date(df)
df = df.dropna(subset=['NACCID', 'visit_date']).copy()
df = df.sort_values(['NACCID', 'visit_date']).reset_index(drop=True)

if 'FORMVER' in df.columns:
    df['FORMVER'] = pd.to_numeric(df['FORMVER'], errors='coerce')

print('Rows:', len(df), 'Participants:', df['NACCID'].nunique())
df[['NACCID','visit_date'] + ([ 'FORMVER' ] if 'FORMVER' in df.columns else [])].head()

C:\Users\branjan\AppData\Local\Temp\ipykernel_24868\1939311393.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['visit_date'] = make_visit_date(df)


Rows: 207454 Participants: 55268


,NACCID,visit_date,FORMVER
0,NACC000011,2006-04-17,1.0
1,NACC000011,2007-06-18,1.0
2,NACC000011,2008-06-03,2.0
3,NACC000011,2009-08-03,2.0
4,NACC000034,2015-07-16,3.0


## 2) Optional FORMVER filter


In [90]:
USE_ONLY_V3 = False 

if USE_ONLY_V3 and 'FORMVER' in df.columns:
    before = len(df)
    df = df[df['FORMVER'].isin([3, 3.2])].copy()
    df = df.sort_values(['NACCID', 'visit_date']).reset_index(drop=True)
    print('Filtered to FORMVER in {3,3.2}: rows', before, '->', len(df))
else:
    print('No FORMVER filtering applied.')

No FORMVER filtering applied.


## 3) Baseline selection
Baseline is the earliest visit where:
- cognitive status is non-demented (CN/MCI-ish), and
- DEMENTED==0 (if available)


In [91]:
import numpy as np
import pandas as pd

def first_row_per_group(df_: pd.DataFrame, group_col: str, sort_col: str) -> pd.DataFrame:
    return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()

# DEMENTED: 0/1
no_dem_mask = (df["DEMENTED"] == 0) if "DEMENTED" in df.columns else pd.Series(True, index=df.index)

MISSING_CODES = {-4, 9}

if "NACCUDSD" in df.columns:
    # Primary baseline definition: CN/MCI (not dementia)
    uds_non_dem = df["NACCUDSD"].isin([1, 2, 3])

    # Fallback for rows where NACCUDSD is missing or coded as somewhat missing.
    if "COGSTAT" in df.columns:
        uds_bad = df["NACCUDSD"].isna() | df["NACCUDSD"].isin(list(MISSING_CODES))
        cog_non_dem = df["COGSTAT"].isin([1, 2, 3, 4])  # clinician judged non-demented range
        cnmci_mask = uds_non_dem | (uds_bad & cog_non_dem)
    else:
        cnmci_mask = uds_non_dem

else:
    # This code uses NORMCOG / NACCTMCI If NACCUDSD is not present
    cn_mask  = (df["NORMCOG"] == 1) if "NORMCOG" in df.columns else pd.Series(False, index=df.index)
    mci_mask = df["NACCTMCI"].isin([1,2,3,4]) if "NACCTMCI" in df.columns else pd.Series(False, index=df.index)

    cnmci_mask = cn_mask | mci_mask

    # Fallback for if even those are missing
    if "COGSTAT" in df.columns:
        cog_non_dem = df["COGSTAT"].isin([1,2,3,4])
        cnmci_mask = cnmci_mask | cog_non_dem

baseline_candidates = df[no_dem_mask & cnmci_mask].copy()

baseline = first_row_per_group(baseline_candidates, "NACCID", "visit_date").copy()
baseline = baseline.rename(columns={"visit_date": "baseline_date"})

print("Baseline participants:", baseline["NACCID"].nunique())
baseline[["NACCID","baseline_date"]].head()

C:\Users\branjan\AppData\Local\Temp\ipykernel_24868\2371402752.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()


Baseline participants: 38341


,NACCID,baseline_date
0,NACC000011,2006-04-17
1,NACC000034,2015-07-16
2,NACC000067,2012-05-23
3,NACC000073,2016-05-20
4,NACC000138,2023-07-13


In [92]:
def first_row_per_group(df_: pd.DataFrame, group_col: str, sort_col: str) -> pd.DataFrame:
    return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()

if "DEMENTED" in df.columns:
    no_dem_mask = (df["DEMENTED"] == 0)
else:
    no_dem_mask = pd.Series(True, index=df.index)

# Baseline eligibility: NOT demented, and CN/MCI
if "NACCUDSD" in df.columns:
    # Best: UDS-derived cognitive status
    cnmci_mask = df["NACCUDSD"].isin([1, 2, 3])
else:
    # Fallback: use NORMCOG and/or NACCTMCI if present
    cn_mask = (df["NORMCOG"] == 1) if "NORMCOG" in df.columns else pd.Series(False, index=df.index)
    mci_mask = df["NACCTMCI"].isin([1, 2, 3, 4]) if "NACCTMCI" in df.columns else pd.Series(False, index=df.index)

    if ("NORMCOG" in df.columns) or ("NACCTMCI" in df.columns):
        cnmci_mask = cn_mask | mci_mask
    else:
        cnmci_mask = pd.Series(True, index=df.index)

if "COGSTAT" in df.columns:
    cog_ok = ~df["COGSTAT"].isin([0, 9, -4])
    # cnmci_mask = cnmci_mask & cog_ok

baseline_candidates = df[no_dem_mask & cnmci_mask].copy()

baseline = (baseline_candidates
            .sort_values(["NACCID", "visit_date"])
            .groupby("NACCID", as_index=False)
            .first())

baseline["baseline_date"] = baseline["visit_date"]
baseline = baseline.drop(columns=["visit_date"])

print("Baseline participants:", baseline["NACCID"].nunique())
baseline[["NACCID", "baseline_date"]].head()

C:\Users\branjan\AppData\Local\Temp\ipykernel_24868\136695756.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .first())


Baseline participants: 38341


,NACCID,baseline_date
0,NACC000011,2006-04-17
1,NACC000034,2015-07-16
2,NACC000067,2012-05-23
3,NACC000073,2016-05-20
4,NACC000138,2023-07-13


## 4) Follow-up time & first dementia event (no horizon restriction yet)
I am first computing time since baseline for every visit, and then identifying each participant’s first post-baseline dementia visit.


In [93]:
def years_between(a: pd.Series, b: pd.Series) -> pd.Series:
    return (b - a).dt.days / 365.25

keep_ids = set(baseline['NACCID'])
df2 = df[df['NACCID'].isin(keep_ids)].merge(
    baseline[['NACCID','baseline_date']], on='NACCID', how='left'
).copy()

df2['t_years'] = years_between(df2['baseline_date'], df2['visit_date'])
post = df2[df2['t_years'] > 0].copy()

print("=== CHECKPOINT: post ===")
print("baseline N:", baseline["NACCID"].nunique())
print("post rows:", len(post), " post unique IDs:", post["NACCID"].nunique())
print("post visit_date min/max:", post["visit_date"].min(), post["visit_date"].max())
print("t_years min/max:", post["t_years"].min(), post["t_years"].max())

max_follow = post.groupby('NACCID')['t_years'].max().rename('max_follow_years')
baseline2 = baseline.merge(max_follow, on='NACCID', how='left')

print("\n=== CHECKPOINT: follow-up ===")
print("max_follow_years describe:")
display(baseline2["max_follow_years"].describe())
print("eligible @4y:", (baseline2["max_follow_years"] >= 4).sum())
print("eligible @5y:", (baseline2["max_follow_years"] >= 5).sum())

baseline2['max_follow_years'] = baseline2['max_follow_years'].fillna(0)

#DEMENTED: 0-No, 1-Yes
#NACCUDSD:1-Normal, 2-Impaired not MCI, 3-MCI, 4-Dementia
if 'NACCUDSD' in post.columns:
    dem_events_all = post[post['NACCUDSD'] == 4].copy()
    dementia_def = 'NACCUDSD==4'
else:
    dem_events_all = post[post['DEMENTED'] == 1].copy()
    dementia_def = 'DEMENTED==1'

first_dem_ever = (dem_events_all.sort_values(['NACCID','visit_date'])
                           .groupby('NACCID', as_index=False)
                           .first()
                           .rename(columns={'visit_date':'dementia_date'}))

first_dem_ever = first_dem_ever.merge(
    baseline2[['NACCID', 'baseline_date', 'max_follow_years']],
    on='NACCID',
    how='left',
    suffixes=('', '_base')   # prevent baseline_date_x/y
)

# Normalizing baseline_date if it got duplicated:
if 'baseline_date' not in first_dem_ever.columns:
    for c in ['baseline_date_base', 'baseline_date_x', 'baseline_date_y']:
        if c in first_dem_ever.columns:
            first_dem_ever['baseline_date'] = first_dem_ever[c]
            break
        
first_dem_ever['time_to_dem_years'] = years_between(first_dem_ever['baseline_date'], first_dem_ever['dementia_date'])

print("\n=== CHECKPOINT: first_dem_ever ===")
print("first_dem_ever unique IDs:", first_dem_ever["NACCID"].nunique())
print("time_to_dem_years min/median/max:",
      first_dem_ever["time_to_dem_years"].min(),
      first_dem_ever["time_to_dem_years"].median(),
      first_dem_ever["time_to_dem_years"].max())

print('Dementia event definition:', dementia_def)
print('Participants with ANY post-baseline dementia event:', first_dem_ever['NACCID'].nunique())
first_dem_ever[['NACCID','baseline_date','dementia_date','time_to_dem_years']].head()

=== CHECKPOINT: post ===
baseline N: 38341
post rows: 123654  post unique IDs: 27703
post visit_date min/max: 2006-04-03 00:00:00 2025-10-21 00:00:00
t_years min/max: 0.008213552361396304 19.392197125256672

=== CHECKPOINT: follow-up ===
max_follow_years describe:


count    27703.000000
mean         5.499126
std          4.268776
min          0.109514
25%          2.080767
50%          4.301164
75%          7.685147
max         19.392197
Name: max_follow_years, dtype: float64

eligible @4y: 14853
eligible @5y: 12474

=== CHECKPOINT: first_dem_ever ===
first_dem_ever unique IDs: 5502
time_to_dem_years min/median/max: 0.4188911704312115 2.6885694729637235 19.304585900068446
Dementia event definition: NACCUDSD==4
Participants with ANY post-baseline dementia event: 5502


C:\Users\branjan\AppData\Local\Temp\ipykernel_24868\2987371432.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .first()


,NACCID,baseline_date,dementia_date,time_to_dem_years
0,NACC000034,2015-07-16,2021-05-21,5.848049
1,NACC000162,2009-03-10,2017-11-10,8.670773
2,NACC000225,2009-09-23,2017-07-07,7.786448
3,NACC000271,2007-06-20,2009-05-14,1.900068
4,NACC000333,2021-05-13,2024-07-24,3.197810


In [94]:
HORIZON_YEARS = 5  # or 4

window = post[post["t_years"] <= HORIZON_YEARS].copy()

# if "NACCUDSD" in window.columns:
#     dem_events_H = window[window["NACCUDSD"] == 4].copy()
# else:
#     dem_events_H = window[window["DEMENTED"] == 1].copy()

dem_events_H = window[window["DEMENTED"] == 1].copy()

first_dem_within_H = (dem_events_H.sort_values(["NACCID", "visit_date"])
                          .groupby("NACCID", as_index=False)
                          .first()
                          .rename(columns={"visit_date": "dementia_date"}))

# Merging only max follow-up
first_dem_within_H = first_dem_within_H.merge(
    baseline2[["NACCID", "max_follow_years"]],
    on="NACCID", how="left"
)

# Computing time-to-dementia using the baseline_date already in first_dem_within_H
if "baseline_date" not in first_dem_within_H.columns:
    if "baseline_date_x" in first_dem_within_H.columns:
        first_dem_within_H = first_dem_within_H.rename(columns={"baseline_date_x": "baseline_date"})
    elif "baseline_date_y" in first_dem_within_H.columns:
        first_dem_within_H = first_dem_within_H.rename(columns={"baseline_date_y": "baseline_date"})
    else:
        raise KeyError("No baseline_date column found in first_dem_within_H.")

first_dem_within_H["time_to_dem_years"] = years_between(first_dem_within_H["baseline_date"], first_dem_within_H["dementia_date"])

print("Participants with dementia within", HORIZON_YEARS, "y:", first_dem_within_H["NACCID"].nunique())
print("time_to_dem_years min/median/max:",
      first_dem_within_H["time_to_dem_years"].min(),
      first_dem_within_H["time_to_dem_years"].median(),
      first_dem_within_H["time_to_dem_years"].max())

Participants with dementia within 5 y: 3958
time_to_dem_years min/median/max: 0.4188911704312115 2.005475701574264 4.999315537303217


C:\Users\branjan\AppData\Local\Temp\ipykernel_24868\2416824035.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .first()


## 5) Etiology subtype mapping (event-level)
Using primary etiology if available:
- PD_LBD primary: `NACCLBDP == 1`
- Vascular primary: `CVDIF == 1`
- AD primary: `NACCALZP == 1` (or fallback `NACCALZD == 1`)


In [95]:

import numpy as np
import pandas as pd


# def subtype_primary(row):

#     formver = row.get("FORMVER", np.nan)

#     # PD / LBD
#     if pd.notna(row.get("NACCLBDP")) and int(row["NACCLBDP"]) == 1:
#         return "PD_LBD"

#     # Vascular
#     if formver >= 3:
#         if pd.notna(row.get("CVDIF")) and int(row["CVDIF"]) == 1:
#             return "Vascular"
#     else:
#         if pd.notna(row.get("VASCIF")) and int(row["VASCIF"]) == 1:
#             return "Vascular"
#         if pd.notna(row.get("VASCPSIF")) and int(row["VASCPSIF"]) == 1:
#             return "Vascular"

#     # AD
#     if pd.notna(row.get("NACCALZP")) and int(row["NACCALZP"]) == 1:
#         return "AD"
#     if pd.notna(row.get("NACCALZD")) and int(row["NACCALZD"]) == 1:
#         return "AD"

#     return "Other/Unk"

def subtype_primary(row):

    formver = row.get("FORMVER", np.nan)

    # PD / LBD
    if pd.notna(row.get("NACCETPR")) and int(row["NACCETPR"]) == 2:
        return "PD_LBD"

    # Vascular
    if pd.notna(row.get("NACCETPR")) and int(row["NACCETPR"]) == 8:
        return "Vascular"

    # AD
    if pd.notna(row.get("NACCETPR")) and int(row["NACCETPR"]) == 1:
        return "AD"

    return "Other/Unk"

def _as_int(x):
    # safe int conversion for strings/float/NA
    if pd.isna(x):
        return None
    try:
        return int(float(x))
    except Exception:
        return None

def subtype_anymention(row) -> str:
    formver = row.get("FORMVER", np.nan)
    formver = float(formver) if pd.notna(formver) else np.nan

    # PD / LBD (any mention)
    # Etiology present (preferred)
    v = _as_int(row.get("NACCLBDE"))
    if v == 1:
        return "PD_LBD"

    # Vascular (any mention)
    if pd.notna(formver) and formver >= 3:
        # UDS v3+: vascular brain injury etiology present
        v = _as_int(row.get("CVD"))
        if v == 1:
            return "Vascular"
    else:
        # UDS v1.2 / v2: vascular dementia etiology flags (probable/possible)
        v = _as_int(row.get("VASC"))
        if v == 1:
            return "Vascular"
        v = _as_int(row.get("VASCPS"))
        if v == 1:
            return "Vascular"

    # AD (any mention)
    v = _as_int(row.get("NACCALZD"))
    if v == 1:
        return "AD"

    return "Other/Unk"

# Subtype labeling at dementia event (v1/v2/v3 compatible)
# Useing functions subtype_primary() and subtype_anymention() defined above.

def subtype_contributing(row) -> str:
    """Contributing etiology ONLY (used as fallback when primary is Other/Unk)."""
    formver = row.get("FORMVER", np.nan)
    try:
        formver = float(formver) if pd.notna(formver) else np.nan
    except Exception:
        formver = np.nan

    # PD/LBD contributing
    if _as_int(row.get("NACCLBDP")) == 2:
        return "PD_LBD"

    # Vascular contributing
    if pd.notna(formver) and formver >= 3:
        if _as_int(row.get("CVDIF")) == 2:
            return "Vascular"
    else:
        if _as_int(row.get("VASCIF")) == 2:
            return "Vascular"
        if _as_int(row.get("VASCPSIF")) == 2:
            return "Vascular"

    # AD contributing
    if _as_int(row.get("NACCALZP")) == 2:
        return "AD"

    return "Other/Unk"


# Stage 1: Primary
first_dem_ever["subtype_primary_contrib_any"] = first_dem_ever.apply(subtype_primary, axis=1)


print("\nFinal (Primary -> Contrib -> Any) subtype counts:")
display(first_dem_ever["subtype_primary_contrib_any"].value_counts())




Final (Primary -> Contrib -> Any) subtype counts:


subtype_primary_contrib_any
AD           4218
Other/Unk     651
PD_LBD        374
Vascular      259
Name: count, dtype: int64

# Section 2D: Trying various combination of features as experiment to generate separate cohot files and then use them in modelling code to check the output score

In [96]:
import os
import numpy as np
import pandas as pd


HORIZON_YEARS = 5
H = HORIZON_YEARS

# 1) Dementia within H (event window)
# dem_H = first_dem_ever.loc[
#     first_dem_ever["time_to_dem_years"] <= H,
#     ["NACCID", "dementia_date", "time_to_dem_years",
#      "subtype_primary_contrib_any", "subtype_label_source_3tier"]
# ].copy()

dem_H = first_dem_ever.loc[
    first_dem_ever["time_to_dem_years"] <= H,
    ["NACCID", "dementia_date", "time_to_dem_years",
     "subtype_primary_contrib_any"]
].copy()


# 2) Merge onto baseline2 and assign labels (old notebook style)
cohort = baseline2.merge(dem_H, on="NACCID", how="left")

cohort["has_dementia_H"] = cohort["dementia_date"].notna()
cohort["has_H_followup"] = cohort["max_follow_years"] >= H

def assign_label(r):
    if r["has_dementia_H"]:
        st = r.get("subtype_primary_contrib_any", "Other/Unk")
        if st in {"AD", "PD_LBD", "Vascular"}:
            return st
        return "MixedOrOther"
    if r["has_H_followup"]:
        #return f"NoDementia_{H}y"
        return f"NoDementia_Hy"
    return "Censored"

cohort["label"] = cohort.apply(assign_label, axis=1)

print("Cohort rows:", len(cohort))
display(cohort["label"].value_counts(dropna=False))

# 3) Build training export X
STARTER_FEATURES = [
    "NACCID","baseline_date",
    "NACCAGE","SEX","RACE","HISPANIC","EDUC","NACCUDSD",

    # Vascular risk / history (baseline-safe, non-leaky)
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
    "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",

    # Additional vascular etiology / imaging / focal findings
    "CVD","CVDIF","VASC","VASCIF",
    "CVDCOG","CVDIMAG","IMAGMWMH","IMAGEWMH",
    "FOCLSYM","FOCLSIGN","HACHIN",

    # Medication proxies for vascular risk treatment (baseline-safe)
    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",

    # Other helpful baseline covariates
    "APNEA","DEP2YRS",

    # Cognition / function
    "NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON",
    "CDRSUM","CDRGLOB",

        # PD/LBD symptom block (UDS B9 clinician judgment)
    "COGFLUC",   # fluctuating cognition (0/1/9) :contentReference[oaicite:0]{index=0}
    "BEVHALL",   # visual hallucinations (0/1/9) :contentReference[oaicite:1]{index=1}
    "BEREM",     # REM sleep behavior disorder (0/1/9) :contentReference[oaicite:5]{index=5}
    # Motor / Parkinsonism exam block (UDS B8)
    "PARKSIGN","BRADY","RESTTRL","RESTTRR","RIGIDL","RIGIDR","SLOWINGL","SLOWINGR","PARKGAIT","GAITDIS",
    "NACCMOM","NACCDAD","SIVDFIND",
]


cols = [c for c in STARTER_FEATURES if c in baseline.columns]
X = baseline[cols].copy()

# Merging label and follow-up info onto X
X = X.merge(
    cohort[["NACCID","baseline_date","label","max_follow_years","has_H_followup","has_dementia_H"]],
    on=["NACCID","baseline_date"],
    how="left"
)


# 4A) UDS missing-code cleansing

# Common UDS special missing / not-applicable codes
# -4 is the big one; 8/9 often used for NA/Unknown for coded fields.
GLOBAL_MISSING = {-4}

# For many binary/coded clinical vars: 8/9 represent NA/Unknown (dictionary-dependent).
CODED_MISSING = {-4, 8, 9}

NUMERIC_TEST_EXTRA_MISSING = {88, 99, 888, 999}

# 1) Force numeric vars to numeric and replace special codes with NaN
# NUM_VARS = ["NACCAGE","EDUC","NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON","CDRSUM","CDRGLOB"]
NUM_VARS = ["NACCAGE","EDUC","NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON","CDRSUM","CDRGLOB","HACHIN"]
for v in NUM_VARS:
    if v in X.columns:
        X[v] = pd.to_numeric(X[v], errors="coerce")

        # Always treating -4 as missing
        X.loc[X[v].isin(GLOBAL_MISSING), v] = np.nan

        # For cognitive test-like vars, also treating 88/99/... as missing
        if v in {"NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON","CDRSUM","CDRGLOB","HACHIN"}:
            X.loc[X[v].isin(NUMERIC_TEST_EXTRA_MISSING), v] = np.nan

# 2) For coded/binary medical history + meds vars, replace -4/8/9 with NaN
CODED_VARS = [
    "SEX","RACE","HISPANIC","NACCUDSD",

    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
    "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",

    "CVD","CVDIF","VASC","VASCIF",
    "CVDCOG","CVDIMAG","IMAGMWMH","IMAGEWMH",
    "FOCLSYM","FOCLSIGN",

    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",
    "APNEA","DEP2YRS",
    "COGFLUC","BEVHALL", "BEREM",

    # Motor block
    "PARKSIGN","BRADY","RESTTRL","RESTTRR","RIGIDL","RIGIDR","SLOWINGL","SLOWINGR","PARKGAIT","GAITDIS",
    "NACCMOM","NACCDAD","SIVDFIND",
]

for v in CODED_VARS:
    if v in X.columns:
        X[v] = pd.to_numeric(X[v], errors="coerce")
        X.loc[X[v].isin(CODED_MISSING), v] = np.nan

# 2b) Special recoding for vascular variables

# FOCLSYM / FOCLSIGN are coded 0=absent, 2=present in this cohort
for v in ["FOCLSYM", "FOCLSIGN"]:
    if v in X.columns:
        X[v] = X[v].map({0: 0, 2: 1})

# CVDIF / VASCIF are multi-class etiologic role variables
# Keep raw values, but also create engineered binary flags
for v in ["CVDIF", "VASCIF"]:
    if v in X.columns:
        X[f"{v}_PRIMARY"] = (X[v] == 1).astype(float)
        X.loc[X[v].isna(), f"{v}_PRIMARY"] = np.nan

        X[f"{v}_PRIMARY_OR_CONTRIB"] = X[v].isin([1, 2]).astype(float)
        X.loc[X[v].isna(), f"{v}_PRIMARY_OR_CONTRIB"] = np.nan

# 3) Adding missing indicators for the coded vars too
for v in CODED_VARS:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)



# 4) Clean coded numerics + add missing indicators
for v in ["TRAILB","TRAILA","NACCMMSE","NACCMOCA","CDRSUM"]:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)


NACC_CSF_PATH  = os.path.join(DATA_DIR, "investigator_fcsf_nacc72.csv")

if "NACC_CSF_PATH" in globals() and os.path.exists(NACC_CSF_PATH):
    csf = pd.read_csv(NACC_CSF_PATH, low_memory=False)
    csf["NACCID"] = csf["NACCID"].astype(str)

    for c in ["CSFABETA","CSFPTAU","CSFTTAU"]:
        if c in csf.columns:
            csf[f"{c}_missing"] = csf[c].isna().astype(int)

    def make_csf_date(df_: pd.DataFrame) -> pd.Series:
        y = pd.to_numeric(df_.get("CSFLPYR"), errors="coerce")
        m = pd.to_numeric(df_.get("CSFLPMO"), errors="coerce").fillna(6).astype(int).clip(1,12)
        d = pd.to_numeric(df_.get("CSFLPDY"), errors="coerce").fillna(15).astype(int).clip(1,28)
        return pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")

    csf["csf_date"] = make_csf_date(csf)

    csf = csf.merge(X[["NACCID","baseline_date"]], on="NACCID", how="inner")
    csf = csf.dropna(subset=["csf_date","baseline_date"]).copy()
    csf["abs_days_from_baseline"] = (csf["csf_date"] - csf["baseline_date"]).abs().dt.days

    csf_nearest = csf.sort_values(["NACCID","abs_days_from_baseline"]).groupby("NACCID", as_index=False).first()

    merge_cols = ["NACCID","CSFABETA","CSFPTAU","CSFTTAU","abs_days_from_baseline"]
    merge_cols = [c for c in merge_cols if c in csf_nearest.columns]
    X = X.merge(csf_nearest[merge_cols], on="NACCID", how="left")

    print("Merged CSF. Rows:", len(X))
    if "CSFABETA" in X.columns:
        print("CSFABETA non-null:", X["CSFABETA"].notna().sum())
else:
    print("CSF file not found (or NACC_CSF_PATH not set); skipping CSF merge.")

# Removing unnecessary missing flags added earlier
IMPORTANT_MISSING = [
    "TRAILA",
    "TRAILB",
    "NACCMMSE",
    "NACCMOCA",
    "CDRSUM"
]

for v in IMPORTANT_MISSING:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)

drop_missing_cols = [
    c for c in X.columns
    if c.endswith("_missing") and
       c.replace("_missing","") not in IMPORTANT_MISSING
]

X = X.drop(columns=drop_missing_cols)

# 6) Feature engineering (vascular-focused)

# 6) Feature engineering (clean engineered features)
import numpy as np

def _bin0(df, cols):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return None
    return df[cols].fillna(0)

# Vascular risk burden
vasc_risk_cols = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","CVCHF","CVANGINA","CVBYPASS","CVHVALVE"]
B = _bin0(X, vasc_risk_cols)
if B is not None:
    X["VASC_RISK_COUNT"] = B.sum(axis=1)

# Vascular meds burden
vasc_med_cols = ["NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]
M = _bin0(X, vasc_med_cols)
if M is not None:
    X["VASC_MED_COUNT"] = M.sum(axis=1)

# Stroke/TIA
S = _bin0(X, ["CBSTROKE","CBTIA"])
if S is not None:
    X["STROKE_TIA_ANY"] = (S.sum(axis=1) > 0).astype(int)

# Executive ratio (TrailB / (TrailA+1))
if "TRAILB" in X.columns and "TRAILA" in X.columns:
    X["EXEC_RATIO_TRAILB_TRAILA"] = X["TRAILB"] / (X["TRAILA"] + 1.0)

# Gait
G = _bin0(X, ["GAITDIS"])
if G is not None:
    X["GAIT_ANY"] = (G.sum(axis=1) > 0).astype(int)

# Hachinski threshold features
if "HACHIN" in X.columns:
    X["HACHIN_GE_4"] = np.where(X["HACHIN"].isna(), np.nan, (X["HACHIN"] >= 4).astype(float))
    X["HACHIN_GE_7"] = np.where(X["HACHIN"].isna(), np.nan, (X["HACHIN"] >= 7).astype(float))

# print("Engineered feats present:",
#       [c for c in ["VASC_RISK_COUNT","VASC_MED_COUNT","STROKE_TIA_ANY","EXEC_RATIO_TRAILB_TRAILA","GAIT_ANY"] if c in X.columns])

X["WMH_ANY"] = (
    (X["IMAGMWMH"] > 0) |
    (X["IMAGEWMH"] > 0)
).astype(int)

X["VASCULAR_BURDEN"] = (
    X[[
        "HYPERTEN",
        "DIABETES",
        "CVAFIB",
        "CBSTROKE",
        "CBTIA",
        "FOCLSYM",
        "FOCLSIGN"
    ]]
    .fillna(0)
    .sum(axis=1)
)

X["CARDIO_BURDEN"] = (
    X[[
        "CVCHF",
        "CVANGINA",
        "CVBYPASS",
        "CVANGIO",
        "CVHVALVE",
        "CVOTHR"
    ]]
    .fillna(0)
    .sum(axis=1)
)

X["CEREBROVASC_EVENT"] = (
    (X["CBSTROKE"] == 1) |
    (X["CBTIA"] == 1) |
    (X["FOCLSYM"] == 2) |
    (X["FOCLSIGN"] == 2)
).astype(int)


print("Engineered feats present:",
      [c for c in [
          "VASC_RISK_COUNT","VASC_MED_COUNT","STROKE_TIA_ANY",
          "EXEC_RATIO_TRAILB_TRAILA","GAIT_ANY",
          "CVDIF_PRIMARY","CVDIF_PRIMARY_OR_CONTRIB",
          "VASCIF_PRIMARY","VASCIF_PRIMARY_OR_CONTRIB",
          "HACHIN_GE_4","HACHIN_GE_7"
      ] if c in X.columns])

Cohort rows: 38341


label
Censored         23300
NoDementia_Hy    11083
AD                3013
MixedOrOther       504
PD_LBD             289
Vascular           152
Name: count, dtype: int64

C:\Users\branjan\AppData\Local\Temp\ipykernel_24868\762032491.py:160: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[f"{v}_missing"] = X[v].isna().astype(int)
C:\Users\branjan\AppData\Local\Temp\ipykernel_24868\762032491.py:160: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[f"{v}_missing"] = X[v].isna().astype(int)
C:\Users\branjan\AppData\Local\Temp\ipykernel_24868\762032491.py:160: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance

Merged CSF. Rows: 38341
CSFABETA non-null: 1559
Engineered feats present: ['VASC_RISK_COUNT', 'VASC_MED_COUNT', 'STROKE_TIA_ANY', 'EXEC_RATIO_TRAILB_TRAILA', 'GAIT_ANY', 'CVDIF_PRIMARY', 'CVDIF_PRIMARY_OR_CONTRIB', 'VASCIF_PRIMARY', 'VASCIF_PRIMARY_OR_CONTRIB', 'HACHIN_GE_4', 'HACHIN_GE_7']


3.1 Vascular Composite Engineering (E8)
Raw vascular risk variables were replaced with:
•	VASC_RISK_COUNT
•	VASC_MED_COUNT
•	STROKE_TIA_ANY
This reduced multicollinearity and improved balanced accuracy slightly (~+0.002–0.003).
Conclusion:
Aggregated vascular burden provided modest improvement but did not dramatically improve subtype separability.
________________________________________
3.2 Executive Dysfunction Ratio (E9)
Created:
EXEC_RATIO_TRAILB_TRAILA
Raw Trails scores removed.
Result:
•	No improvement in balanced accuracy.
•	Slight decrease in vascular recall.
Conclusion:
Executive dysfunction ratio did not meaningfully improve subtype discrimination.
________________________________________
3.3 Executive-Memory Axis (E10)
Constructed:
EXEC_MINUS_MEM
Using train-only z-scored:
•	Trail B (executive)
•	MMSE/MOCA (memory)
Result:
•	Performance similar to baseline.
•	No meaningful vascular improvement.
Conclusion:
Linear combination of executive-memory features did not enhance subtype separability.
________________________________________
3.4 Addition of CSF Biomarkers
CSFABETA, CSFPTAU, CSFTTAU added to E8.
Result:
•	Balanced accuracy ~0.632 (no material change)
•	No vascular recall improvement
Conclusion:
CSF biomarkers did not improve multi-class subtype discrimination at baseline, likely due to overlapping biomarker profiles and limited vascular coverage.


In [97]:
# =========================
# Vascular feature presence analysis
# =========================

# 20 vascular variables to evaluate
vascular_vars = [
    "CVD",
    "CVDIF",
    "VASC",
    "VASCIF",
    "CBSTROKE",
    "CBTIA",
    "STROKMUL",
    "HYPERTEN",
    "DIABETES",
    "HYPERCHO",
    "CVAFIB",
    "CVDIMAG",
    "IMAGMWMH",
    "IMAGEWMH",
    "FOCLSYM",
    "FOCLSIGN",
    "HACHIN",
    "CVDCOG",
    "STROKCOG",
    "INFWMH",
]

# We want these vars in X if available in baseline
# Add any missing ones from baseline now (safe merge-by-column approach)
extra_vasc_cols = [c for c in vascular_vars if c in baseline.columns and c not in X.columns]
if extra_vasc_cols:
    tmp = baseline[["NACCID", "baseline_date"] + extra_vasc_cols].copy()
    X = X.merge(tmp, on=["NACCID", "baseline_date"], how="left")
    print("Added extra vascular columns into X:", extra_vasc_cols)
else:
    print("No additional vascular columns needed; all available ones already in X.")

# Clean these newly added vascular vars using your same missing-code logic
for v in vascular_vars:
    if v in X.columns:
        X[v] = pd.to_numeric(X[v], errors="coerce")

        # Default cleaning for coded vars
        if v in {"CVDIF", "VASCIF"}:
            # these are not simple binary flags; keep valid category codes, only remove structural missing
            X.loc[X[v].isin(GLOBAL_MISSING), v] = np.nan
        elif v == "HACHIN":
            # numeric score; in older forms 88/99 type codes may appear
            X.loc[X[v].isin(GLOBAL_MISSING | NUMERIC_TEST_EXTRA_MISSING), v] = np.nan
        else:
            # most others are coded yes/no-ish variables
            X.loc[X[v].isin(CODED_MISSING), v] = np.nan

# What vascular vars actually exist?
existing_vascular_vars = [v for v in vascular_vars if v in X.columns]
missing_vascular_vars = [v for v in vascular_vars if v not in X.columns]

print("\nExisting vascular vars in X:")
print(existing_vascular_vars)

print("\nMissing vascular vars (not available in this cohort build):")
print(missing_vascular_vars)

# --------
# Overall population summary
# --------
summary_rows = []

for v in existing_vascular_vars:
    s = X[v]

    row = {
        "variable": v,
        "n_total": len(X),
        "n_missing": int(s.isna().sum()),
        "pct_missing": float(s.isna().mean()),
        "n_nonmissing": int(s.notna().sum()),
        "n_unique_nonmissing": int(s.dropna().nunique()),
    }

    # common binary-ish counts
    row["n_0"] = int((s == 0).sum())
    row["n_1"] = int((s == 1).sum())

    # extra counts for non-binary / categorical vars if present
    for val in [2, 3, 7, 8]:
        row[f"n_{val}"] = int((s == val).sum())

    summary_rows.append(row)

vascular_summary_df = pd.DataFrame(summary_rows).sort_values(
    ["n_1", "n_nonmissing"], ascending=[False, False]
)

print("\nOverall vascular variable summary:")
display(vascular_summary_df)

# --------
# By-label comparison
# --------
# This is the key part for model usefulness:
# a variable helps Vascular vs others only if its distribution differs by label.

label_order = ["AD", "PD_LBD", "Vascular", "MixedOrOther", "NoDementia_Hy", "Censored"]
label_order = [x for x in label_order if x in X["label"].dropna().unique().tolist()]

compare_rows = []

for v in existing_vascular_vars:
    for lab in label_order:
        sub = X.loc[X["label"] == lab, v]

        row = {
            "variable": v,
            "label": lab,
            "n_rows": len(sub),
            "n_missing": int(sub.isna().sum()),
            "pct_missing": float(sub.isna().mean()) if len(sub) else np.nan,
            "n_0": int((sub == 0).sum()),
            "n_1": int((sub == 1).sum()),
            "rate_1_among_nonmissing": float((sub == 1).sum() / sub.notna().sum()) if sub.notna().sum() > 0 else np.nan,
        }

        for val in [2, 3, 7, 8]:
            row[f"n_{val}"] = int((sub == val).sum())

        compare_rows.append(row)

vascular_by_label_df = pd.DataFrame(compare_rows)

print("\nVascular variable counts by label:")
display(vascular_by_label_df)

# --------
# Vascular vs Others enrichment table
# --------
# This helps rank which features are actually enriched in vascular cases.
signal_rows = []

vascular_only = X[X["label"] == "Vascular"].copy()
others_only = X[X["label"] != "Vascular"].copy()

for v in existing_vascular_vars:
    sv = vascular_only[v]
    so = others_only[v]

    vascular_rate_1 = float((sv == 1).sum() / sv.notna().sum()) if sv.notna().sum() > 0 else np.nan
    other_rate_1 = float((so == 1).sum() / so.notna().sum()) if so.notna().sum() > 0 else np.nan

    signal_rows.append({
        "variable": v,
        "vascular_n": len(sv),
        "others_n": len(so),
        "vascular_nonmissing": int(sv.notna().sum()),
        "others_nonmissing": int(so.notna().sum()),
        "vascular_rate_1": vascular_rate_1,
        "other_rate_1": other_rate_1,
        "rate_diff": vascular_rate_1 - other_rate_1 if pd.notna(vascular_rate_1) and pd.notna(other_rate_1) else np.nan,
        "rate_ratio": (vascular_rate_1 / (other_rate_1 + 1e-6)) if pd.notna(vascular_rate_1) and pd.notna(other_rate_1) else np.nan,
    })

vascular_signal_df = pd.DataFrame(signal_rows).sort_values(
    ["rate_diff", "rate_ratio"], ascending=[False, False]
)

print("\nTop vascular-enriched variables (best candidates for improving vascular recall):")
display(vascular_signal_df)

# Optional: compact quick tables for only AD / PD_LBD / Vascular
focus_labels = [x for x in ["AD", "PD_LBD", "Vascular"] if x in X["label"].unique()]
if focus_labels:
    focused_df = vascular_by_label_df[vascular_by_label_df["label"].isin(focus_labels)].copy()
    print("\nFocused comparison for AD vs PD_LBD vs Vascular:")
    display(focused_df)

Added extra vascular columns into X: ['STROKMUL', 'STROKCOG', 'INFWMH']

Existing vascular vars in X:
['CVD', 'CVDIF', 'VASC', 'VASCIF', 'CBSTROKE', 'CBTIA', 'STROKMUL', 'HYPERTEN', 'DIABETES', 'HYPERCHO', 'CVAFIB', 'CVDIMAG', 'IMAGMWMH', 'IMAGEWMH', 'FOCLSYM', 'FOCLSIGN', 'HACHIN', 'CVDCOG', 'STROKCOG', 'INFWMH']

Missing vascular vars (not available in this cohort build):
[]

Overall vascular variable summary:


,variable,n_total,n_missing,pct_missing,n_nonmissing,n_unique_nonmissing,n_0,n_1,n_2,n_3,n_7,n_8
9,HYPERCHO,38341,633,0.016510,37708,3,18149,18405,1154,0,0,0
7,HYPERTEN,38341,360,0.009389,37981,3,18887,18106,988,0,0,0
16,HACHIN,38341,17865,0.465950,20476,12,9087,9030,677,788,118,31
8,DIABETES,38341,356,0.009285,37985,3,32447,5237,301,0,0,0
10,CVAFIB,38341,430,0.011215,37911,3,35500,1762,649,0,0,0
15,FOCLSIGN,38341,17865,0.465950,20476,2,19247,1229,0,0,0,0
0,CVD,38341,21022,0.548290,17319,2,16190,1129,0,0,0,0
12,IMAGMWMH,38341,30100,0.785060,8241,2,7343,898,0,0,0,0
14,FOCLSYM,38341,17865,0.465950,20476,2,19604,872,0,0,0,0
17,CVDCOG,38341,32070,0.836441,6271,2,5763,508,0,0,0,0



Vascular variable counts by label:


,variable,label,n_rows,n_missing,pct_missing,n_0,n_1,rate_1_among_nonmissing,n_2,n_3,n_7,n_8
0,CVD,AD,3013,2116,0.702290,813,84,0.093645,0,0,0,0
1,CVD,PD_LBD,289,169,0.584775,116,4,0.033333,0,0,0,0
2,CVD,Vascular,152,115,0.756579,13,24,0.648649,0,0,0,0
3,CVD,MixedOrOther,504,328,0.650794,159,17,0.096591,0,0,0,0
4,CVD,NoDementia_Hy,11083,8036,0.725074,2927,120,0.039383,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
115,INFWMH,PD_LBD,289,285,0.986159,3,1,0.250000,0,0,0,0
116,INFWMH,Vascular,152,133,0.875000,14,5,0.263158,0,0,0,0
117,INFWMH,MixedOrOther,504,488,0.968254,15,1,0.062500,0,0,0,0
118,INFWMH,NoDementia_Hy,11083,10986,0.991248,88,9,0.092784,0,0,0,0



Top vascular-enriched variables (best candidates for improving vascular recall):


,variable,vascular_n,others_n,vascular_nonmissing,others_nonmissing,vascular_rate_1,other_rate_1,rate_diff,rate_ratio
11,CVDIMAG,152,38189,30,3785,0.733333,0.069485,0.663849,10.553714
17,CVDCOG,152,38189,36,6235,0.666667,0.077626,0.589040,8.588044
0,CVD,152,38189,37,17282,0.648649,0.063939,0.584709,10.144589
1,CVDIF,152,38189,33,6754,0.636364,0.062185,0.574178,10.233169
18,STROKCOG,152,38189,19,1925,0.473684,0.050909,0.422775,9.304329
12,IMAGMWMH,152,38189,23,8218,0.434783,0.108055,0.326727,4.023660
7,HYPERTEN,152,38189,150,37831,0.760000,0.475589,0.284411,1.598016
13,IMAGEWMH,152,38189,22,8201,0.227273,0.019022,0.208251,11.947216
2,VASC,152,38189,87,8540,0.218391,0.018970,0.199421,11.512094
3,VASCIF,152,38189,87,8540,0.183908,0.014637,0.169271,12.563739



Focused comparison for AD vs PD_LBD vs Vascular:


,variable,label,n_rows,n_missing,pct_missing,n_0,n_1,rate_1_among_nonmissing,n_2,n_3,n_7,n_8
0,CVD,AD,3013,2116,0.702290,813,84,0.093645,0,0,0,0
1,CVD,PD_LBD,289,169,0.584775,116,4,0.033333,0,0,0,0
2,CVD,Vascular,152,115,0.756579,13,24,0.648649,0,0,0,0
6,CVDIF,AD,3013,2178,0.722868,0,24,0.028743,54,6,751,0
7,CVDIF,PD_LBD,289,171,0.591696,0,2,0.016949,1,1,114,0
8,CVDIF,Vascular,152,119,0.782895,0,21,0.636364,2,0,10,0
12,VASC,AD,3013,1191,0.395287,1803,19,0.010428,0,0,0,0
13,VASC,PD_LBD,289,138,0.477509,148,3,0.019868,0,0,0,0
14,VASC,Vascular,152,65,0.427632,68,19,0.218391,0,0,0,0
18,VASCIF,AD,3013,1191,0.395287,0,12,0.006586,7,0,1803,0


In [98]:
for v in ["FOCLSYM", "FOCLSIGN", "HACHIN", "CVDIF", "VASCIF"]:
    if v in X.columns:
        print("\n", v)
        print(X[v].value_counts(dropna=False).sort_index())


 FOCLSYM
FOCLSYM
0.0    19604
1.0      872
NaN    17865
Name: count, dtype: int64

 FOCLSIGN
FOCLSIGN
0.0    19247
1.0     1229
NaN    17865
Name: count, dtype: int64

 HACHIN
HACHIN
0.0      9087
1.0      9030
2.0       677
3.0       788
4.0       265
5.0       324
6.0        77
7.0       118
8.0        31
9.0        43
10.0       33
11.0        3
NaN     17865
Name: count, dtype: int64

 CVDIF
CVDIF
1.0      441
2.0      367
3.0       81
7.0     5898
NaN    31554
Name: count, dtype: int64

 VASCIF
VASCIF
1.0      141
2.0       40
7.0     8446
NaN    29714
Name: count, dtype: int64


In [99]:
for v in ["FOCLSYM", "FOCLSIGN", "HACHIN", "CVDIF", "VASCIF"]:
    if v in X.columns:
        print("\n===================")
        print(v)
        print(pd.crosstab(X["label"], X[v], dropna=False))


FOCLSYM
FOCLSYM         0.0  1.0    NaN
label                          
AD             2025   65    923
Censored       9524  516  13260
MixedOrOther    281   43    180
NoDementia_Hy  7537  205   3341
PD_LBD          146   21    122
Vascular         91   22     39

FOCLSIGN
FOCLSIGN        0.0  1.0    NaN
label                          
AD             1952  138    923
Censored       9314  726  13260
MixedOrOther    270   54    180
NoDementia_Hy  7483  259   3341
PD_LBD          138   29    122
Vascular         90   23     39

HACHIN
HACHIN         0.0   1.0   2.0   3.0   4.0   5.0   6.0   7.0   8.0   9.0   10.0  11.0   NaN 
label                                                                                       
AD              826   987    85   107    24    36     4    13     1     4     3     0    923
Censored       4114  4549   378   459   159   192    50    75    20    28    15     1  13260
MixedOrOther    124   115    24    18    20    18     3     1     0     1     0     0    

In [121]:
# 7) Save outputs
#out_full = f"cohort_full_H{H}.csv"   # full baseline2 + label + event info (debug)
out_XH   = f"cohort_H{H}.csv"        # training export for this horizon
out_X    = "cohort.csv"            

#out_full_path = os.path.join(OUT_DIR, out_full)
out_XH_path = os.path.join(OUT_DIR, out_XH)
out_X_path = os.path.join(OUT_DIR, out_X)

# cohort.to_csv(out_full_path, index=False)
X.to_csv(out_XH_path, index=False)
X.to_csv(out_X_path, index=False)

# print("Wrote:", out_full, out_XH, "and", out_X)
print("Training export label counts:")
display(X["label"].value_counts(dropna=False))



# 7) Save outputs (single + per-experiment files)
import json

BASE_COLS = [
    "NACCID", "baseline_date",
    "label", "max_follow_years", "has_H_followup", "has_dementia_H"
]

# Defining experiment feature groups
DEMOGRAPHIC = ["NACCAGE","SEX","RACE","HISPANIC","EDUC","NACCUDSD"]

# Cognitive + function (+ missing flags)
COGNITIVE = [
    "NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON",
    "CDRSUM","CDRGLOB",
    "NACCMMSE_missing","NACCMOCA_missing","TRAILA_missing","TRAILB_missing","CDRSUM_missing"
]

# Medical / vascular block
MEDICAL_VASCULAR = [
    # baseline vascular risk/history
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
    "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",

    # vascular-specific diagnosis / imaging / neuro signs
    "CVD","VASC","CVDCOG","CVDIMAG","IMAGMWMH","IMAGEWMH",
    "FOCLSYM","FOCLSIGN","HACHIN",

    # etiologic role engineered flags
    "CVDIF_PRIMARY","CVDIF_PRIMARY_OR_CONTRIB",
    "VASCIF_PRIMARY","VASCIF_PRIMARY_OR_CONTRIB",

    # other medical
    "APNEA","DEP2YRS",

    # meds / treatment proxies
    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"
]

# Motor / PD-LBD exam + symptom block
MOTOR_PD = [
    "COGFLUC","BEVHALL","BEREM",
    "PARKSIGN","BRADY","RESTTRL","RESTTRR","RIGIDL","RIGIDR",
    "SLOWINGL","SLOWINGR","PARKGAIT","GAITDIS"
]

# CSF biomarkers
CSF = ["CSFABETA","CSFPTAU","CSFTTAU","abs_days_from_baseline"]  
VASC_ENGINEERED = [
    "VASC_RISK_COUNT",
    "STROKE_TIA_ANY",
    "VASC_MED_COUNT",
    "GAIT_ANY",

    # extra vascular engineered features
    "HACHIN_GE_4",
    "HACHIN_GE_7",
    # "CVDIF_PRIMARY",
    # "CVDIF_PRIMARY_OR_CONTRIB",
    # "VASCIF_PRIMARY",
    # "VASCIF_PRIMARY_OR_CONTRIB",
]




EXEC_ENGINEERED = ["EXEC_RATIO_TRAILB_TRAILA"]
EXTRA = ["NACCMOM", "NACCDAD","SIVDFIND"]

# EXPERIMENTS = {
#     # "E0_demo": DEMOGRAPHIC,
#     # "E1_demo_cog": DEMOGRAPHIC + COGNITIVE,
#     # "E2_demo_cog_med": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR,
#     # "E3_demo_cog_motor": DEMOGRAPHIC + COGNITIVE + MOTOR_PD,
#     # "E4_full_clinical": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR + MOTOR_PD,
#     # # Below are some of the CSF subset experiments
#     # "E5_full_plus_csf": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR + MOTOR_PD + CSF,
#     # "E6_demo_csf": DEMOGRAPHIC + CSF,
#     # "E7_demo_med": DEMOGRAPHIC + MEDICAL_VASCULAR,
#     # E8 cleaning vascular
#     "E8_clean_vascular": DEMOGRAPHIC + COGNITIVE + MOTOR_PD + VASC_ENGINEERED + EXTRA,

#     # E9 cleaning exec ratio: use ratio, drop raw Trails later
#     # "E9_clean_exec_ratio": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR + MOTOR_PD + EXEC_ENGINEERED,

#     # E10 executive-memory axis: will use engineered EXEC_MINUS_MEM + MEM_SCORE
#     # "E10_exec_mem_axis_raw": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR + MOTOR_PD,
# }

# VASC_RISK_COUNT = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","CVCHF","CVANGINA","CVBYPASS","CVHVALVE"]


# # Vascular meds burden
# VASC_MED_COUNT = ["NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]


# # Stroke/TIA
# STROKE_TIA_ANY = _bin0(X, ["CBSTROKE","CBTIA"])

EXPERIMENTS = {
    # Main vascular experiments

    # E8a: high-coverage vascular features only
    "E8a_vascular_coverage": (
        DEMOGRAPHIC
        + COGNITIVE
        + MOTOR_PD
        + [
            "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
            "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",
            "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",
            "APNEA","DEP2YRS",
            "VASC_RISK_COUNT","STROKE_TIA_ANY","VASC_MED_COUNT","GAIT_ANY"
        ]
        + EXTRA
    ),

    # E8b: high-coverage + high-signal vascular variables
    "E8b_vascular_signal": (
        DEMOGRAPHIC
        + COGNITIVE
        + MOTOR_PD
        + [
            "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
             "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",
            # "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",
            "APNEA","DEP2YRS",

            "CVD","VASC","CVDCOG","CVDIMAG","IMAGMWMH","IMAGEWMH",
            "FOCLSYM","FOCLSIGN","HACHIN",
            "HACHIN_GE_4","HACHIN_GE_7",
            # "CVDIF_PRIMARY","CVDIF_PRIMARY_OR_CONTRIB",
            # "VASCIF_PRIMARY","VASCIF_PRIMARY_OR_CONTRIB",
            #"CEREBROVASC_EVENT",
            #"VASCULAR_BURDEN",
            #"VASC_RISK_COUNT",
            "STROKE_TIA_ANY",
            "VASC_MED_COUNT",
            "GAIT_ANY",
            #"WMH_ANY",
            #"CARDIO_BURDEN",
        ]
        + EXTRA
    ),

    # Optional fuller clinical model with refined vascular block
     "E8c_full_clinical_vascular": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR + MOTOR_PD + VASC_ENGINEERED + EXTRA,
}

# Select columns safely 
def select_cols(df_, cols):
    cols = list(dict.fromkeys(cols))  # de-dupe, preserve order
    present = [c for c in cols if c in df_.columns]
    missing = [c for c in cols if c not in df_.columns]
    return present, missing

out_XH   = f"cohort_H{H}.csv"
out_X    = "cohort.csv"

out_XH_path = os.path.join(OUT_DIR, out_XH)
out_X_path  = os.path.join(OUT_DIR, out_X)

X.to_csv(out_XH_path, index=False)
X.to_csv(out_X_path, index=False)

print("Wrote:", out_XH, "and", out_X)
print("Training export label counts:")
display(X["label"].value_counts(dropna=False))

manifest_rows = []
exp_dir = OUT_DIR 

for exp_name, feat_list in EXPERIMENTS.items():
    wanted = BASE_COLS + feat_list
    present, missing = select_cols(X, wanted)

    # keeping only columns that exist
    X_exp = X[present].copy()

    # saving file
    exp_file = f"cohort_{exp_name}_H{H}.csv"
    exp_path = os.path.join(exp_dir, exp_file)
    X_exp.to_csv(exp_path, index=False)

    manifest_rows.append({
        "experiment": exp_name,
        "file": exp_file,
        "n_rows": len(X_exp),
        "n_cols": X_exp.shape[1],
        "missing_cols": ";".join(missing) if missing else "",
        "features_json": json.dumps(feat_list),
    })

    if missing:
        print(f"[{exp_name}] wrote {exp_file} (NOTE: missing {len(missing)} cols: {missing[:8]}{'...' if len(missing)>8 else ''})")
    else:
        print(f"[{exp_name}] wrote {exp_file}")

manifest = pd.DataFrame(manifest_rows)
manifest_file = f"cohort_manifest_H{H}.csv"
manifest_path = os.path.join(exp_dir, manifest_file)
manifest.to_csv(manifest_path, index=False)
print("Wrote manifest:", manifest_file)

display(manifest[["experiment","file","n_rows","n_cols","missing_cols"]])


Training export label counts:


label
Censored         23300
NoDementia_Hy    11083
AD                3013
MixedOrOther       504
PD_LBD             289
Vascular           152
Name: count, dtype: int64

Wrote: cohort_H5.csv and cohort.csv
Training export label counts:


label
Censored         23300
NoDementia_Hy    11083
AD                3013
MixedOrOther       504
PD_LBD             289
Vascular           152
Name: count, dtype: int64

[E8a_vascular_coverage] wrote cohort_E8a_vascular_coverage_H5.csv
[E8b_vascular_signal] wrote cohort_E8b_vascular_signal_H5.csv
[E8c_full_clinical_vascular] wrote cohort_E8c_full_clinical_vascular_H5.csv
Wrote manifest: cohort_manifest_H5.csv


,experiment,file,n_rows,n_cols,missing_cols
0,E8a_vascular_coverage,cohort_E8a_vascular_coverage_H5.csv,38341,66,
1,E8b_vascular_signal,cohort_E8b_vascular_signal_H5.csv,38341,69,
2,E8c_full_clinical_vascular,cohort_E8c_full_clinical_vascular_H5.csv,38341,81,


In [122]:
print("Number of modeling columns:", len(X.columns))
print(sorted(X.columns))

Number of modeling columns: 95
['ANIMALS', 'APNEA', 'BEREM', 'BEVHALL', 'BOSTON', 'BRADY', 'CARDIO_BURDEN', 'CBSTROKE', 'CBTIA', 'CDRGLOB', 'CDRSUM', 'CDRSUM_missing', 'CEREBROVASC_EVENT', 'COGFLUC', 'CSFABETA', 'CSFPTAU', 'CSFTTAU', 'CVAFIB', 'CVANGINA', 'CVANGIO', 'CVBYPASS', 'CVCHF', 'CVD', 'CVDCOG', 'CVDIF', 'CVDIF_PRIMARY', 'CVDIF_PRIMARY_OR_CONTRIB', 'CVDIMAG', 'CVHVALVE', 'CVOTHR', 'DEP2YRS', 'DIABETES', 'EDUC', 'EXEC_RATIO_TRAILB_TRAILA', 'FOCLSIGN', 'FOCLSYM', 'GAITDIS', 'GAIT_ANY', 'HACHIN', 'HACHIN_GE_4', 'HACHIN_GE_7', 'HISPANIC', 'HYPERCHO', 'HYPERTEN', 'IMAGEWMH', 'IMAGMWMH', 'INFWMH', 'NACCAC', 'NACCAGE', 'NACCANGI', 'NACCBETA', 'NACCCCBS', 'NACCDAD', 'NACCDIUR', 'NACCID', 'NACCLIPL', 'NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'NACCMOM', 'NACCUDSD', 'NACCVASD', 'PARKGAIT', 'PARKSIGN', 'RACE', 'RESTTRL', 'RESTTRR', 'RIGIDL', 'RIGIDR', 'SEX', 'SIVDFIND', 'SLOWINGL', 'SLOWINGR', 'STROKCOG', 'STROKE_TIA_ANY', 'STROKMUL', 'TRAILA', 'TRAILA_missing', 'TRAI